# 1. 读取全队列的作者元数据

之前的参考只使用CID4535，这里先统计Wu全队列各供者的细胞覆盖，为后续划分参考和保留供者做准备。

当前使用作者orig.ident作为供者/样本标识，保留原名和后缀；subtype使用作者提供的亚型。统计范围是全队列作者发布的细胞元数据，CID4535使用全队列中的记录。


In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd
from IPython.display import display

project_dir = Path("/Users/georicl/Documents/sc_sncell_github")
input_dir = project_dir / "data/raw/wu2021_scrna/Wu_etal_2021_BRCA_scRNASeq"
output_dir = project_dir / "results/Wu2021_donor_reference_inventory"
metadata_path = input_dir / "metadata.csv"
barcode_path = input_dir / "count_matrix_barcodes.tsv"

metadata = pd.read_csv(metadata_path, index_col=0)
barcodes = pd.read_csv(barcode_path, sep="\t", header=None)[0].astype(str)
required_columns = ["orig.ident", "subtype", "celltype_major", "celltype_minor"]
assert metadata.index.is_unique
assert not metadata[required_columns].isna().any().any()
assert barcodes.is_unique
assert set(barcodes) == set(metadata.index)
metadata = metadata.loc[barcodes].copy()

print("细胞数量：", len(metadata))
print("作者样本标识数量：", metadata["orig.ident"].nunique())
display(metadata[required_columns].head())


细胞数量： 100064
作者样本标识数量： 26


,orig.ident,subtype,celltype_major,celltype_minor
CID3586_AAGACCTCAGCATGAG,CID3586,HER2+,Endothelial,Endothelial ACKR1
CID3586_AAGGTTCGTAGTACCT,CID3586,HER2+,Endothelial,Endothelial ACKR1
CID3586_ACCAGTAGTTGTGGCC,CID3586,HER2+,Endothelial,Endothelial ACKR1
CID3586_ACCCACTAGATGTCGG,CID3586,HER2+,Endothelial,Endothelial ACKR1
CID3586_ACTGATGGTCAACTGT,CID3586,HER2+,Endothelial,Endothelial ACKR1


# 2. 按当前规则建立参考标签

与CID4535的第一次RCTD保持一致，其他细胞沿用celltype_major；T-cells使用celltype_minor区分CD4、CD8、NK、NKT和Cycling T。作者原标签和小群体的细胞记录都保留。


In [2]:
metadata["reference_label"] = metadata["celltype_major"].astype("string")
is_t_group = metadata["celltype_major"].eq("T-cells")
metadata.loc[is_t_group, "reference_label"] = (
    metadata.loc[is_t_group, "celltype_minor"].astype("string")
)

reference_types = [
    "CAFs", "T cells CD8+", "T cells CD4+", "NK cells", "NKT cells",
    "Cycling T-cells", "PVL", "Cancer Epithelial", "Normal Epithelial",
    "Endothelial", "Myeloid", "B-cells", "Plasmablasts",
]
assert set(metadata["reference_label"]) == set(reference_types)

label_mapping = (
    metadata[["celltype_major", "celltype_minor", "reference_label"]]
    .drop_duplicates()
    .sort_values(["celltype_major", "celltype_minor"])
)
display(label_mapping)


,celltype_major,celltype_minor,reference_label
CID3586_AAAGATGCAGGGAGAG,B-cells,B cells Memory,B-cells
CID3586_AAACCTGAGTTACGGG,B-cells,B cells Naive,B-cells
CID3586_ACAGCTAAGACGCACA,CAFs,CAFs MSC iCAF-like,CAFs
CID3586_GATTCAGGTTCCACAA,CAFs,CAFs myCAF-like,CAFs
CID45171_ACCCACTTCCTAGAAC,Cancer Epithelial,Cancer Basal SC,Cancer Epithelial
CID3921_ACATCAGAGGGTGTTG,Cancer Epithelial,Cancer Cycling,Cancer Epithelial
CID3921_AAACGGGGTTACGCGC,Cancer Epithelial,Cancer Her2 SC,Cancer Epithelial
CID4066_ATGAGGGTCATTCACT,Cancer Epithelial,Cancer LumA SC,Cancer Epithelial
CID45171_ACTGAACTCTGTTGAG,Cancer Epithelial,Cancer LumB SC,Cancer Epithelial
CID3586_AAGACCTCAGCATGAG,Endothelial,Endothelial ACKR1,Endothelial


# 3. 生成供者 × 参考细胞类型的数量表

每行对应一个作者样本标识，每列是一种参考类型，0表示该样本在作者发布数据中没有这一类细胞。先核对每个样本是否只对应一种亚型，再加入总细胞数。

这张表统计逐供者QC前的细胞数量。CID4535已经用于流程开发，训练和测试供者的分配在后续记录。


In [3]:
subtype_counts = metadata.groupby("orig.ident")["subtype"].nunique()
assert subtype_counts.eq(1).all(), "存在一个样本对应多种亚型，需要核对"

counts = pd.crosstab(metadata["orig.ident"], metadata["reference_label"])
counts = counts.reindex(columns=reference_types, fill_value=0).astype(int)
counts.index.name = "donor_id"
donor_info = metadata.groupby("orig.ident").agg(
    subtype=("subtype", "first"),
    total_cells=("subtype", "size"),
)
donor_info.index.name = "donor_id"
donor_counts = donor_info.join(counts).sort_values(["subtype", "donor_id"])

assert donor_counts[reference_types].sum(axis=1).equals(donor_counts["total_cells"])
assert donor_counts["total_cells"].sum() == len(metadata)

# 核对CID4535与之前已使用的参考标签数量一致
baseline_path = (
    project_dir / "data/processed/CID4535_combine_v1/rctd_input/reference_metadata.csv"
)
baseline_metadata = pd.read_csv(baseline_path, index_col=0)
expected = baseline_metadata["reference_label"].value_counts().reindex(reference_types, fill_value=0)
assert donor_counts.loc["CID4535", reference_types].astype(int).tolist() == expected.tolist()

with pd.option_context("display.max_rows", 40, "display.max_columns", 20, "display.width", 220):
    display(donor_counts)


,subtype,total_cells,CAFs,T cells CD8+,T cells CD4+,NK cells,NKT cells,Cycling T-cells,PVL,Cancer Epithelial,Normal Epithelial,Endothelial,Myeloid,B-cells,Plasmablasts
donor_id,,,,,,,,,,,,,,,
CID3941,ER+,631,8,126,108,18,9,5,25,196,0,44,37,55,0
CID3948,ER+,2327,15,498,845,58,40,24,62,261,0,85,122,85,232
CID4040,ER+,2531,129,528,830,107,22,25,443,0,0,218,50,105,74
CID4067,ER+,3764,135,243,353,48,39,6,83,2352,0,186,266,53,0
CID4290A,ER+,5789,280,115,345,50,24,8,140,4053,18,298,341,117,0
CID4398,ER+,4451,178,926,2313,288,129,77,87,0,0,101,225,36,91
CID4461,ER+,631,41,11,44,2,3,8,48,207,0,182,53,0,32
CID4463,ER+,1138,25,75,115,3,19,5,31,659,26,79,101,0,0
CID4471,ER+,8609,1292,159,412,30,32,8,1285,212,1966,2778,285,99,51


# 4. 汇总亚型并保存数量表

保存宽表供直接查看，长表供后续绘图和筛选，另外保存作者标签到参考标签的对应关系与输入来源。这里统计现有细胞数量，供后续查看供者的细胞覆盖。


In [4]:
subtype_summary = donor_counts.groupby("subtype").agg(
    n_donor_ids=("total_cells", "size"),
    total_cells=("total_cells", "sum"),
)
subtype_summary = subtype_summary.join(
    donor_counts.groupby("subtype")[reference_types].sum()
)
long_counts = donor_counts.reset_index().melt(
    id_vars=["donor_id", "subtype", "total_cells"],
    value_vars=reference_types,
    var_name="reference_label",
    value_name="n_cells",
)

output_dir.mkdir(parents=True, exist_ok=True)
donor_counts.to_csv(output_dir / "donor_by_reference_celltype.csv")
long_counts.to_csv(output_dir / "donor_by_reference_celltype_long.csv", index=False)
subtype_summary.to_csv(output_dir / "subtype_summary.csv")
label_mapping.to_csv(output_dir / "reference_label_mapping.csv", index=False)

def markdown_table(frame):
    frame = frame.reset_index()
    lines = [
        "| " + " | ".join(map(str, frame.columns)) + " |",
        "| " + " | ".join(["---"] * len(frame.columns)) + " |",
    ]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in frame.itertuples(index=False, name=None)]
    return "\n".join(lines)

report = (
    "# Wu 2021 供者 × 参考细胞类型数量表\n\n"
    f"共{len(donor_counts)}个作者样本标识、{len(metadata):,}个细胞、{len(reference_types)}种参考标签。\n\n"
    "供者列使用作者orig.ident原值；亚型使用subtype，不猜测后缀合并。"
    "统计全队列作者元数据，未追加本项目QC，也未重复加入单样本下载包。"
    "CID4535已用于当前流程开发；本表不自动分配参考或测试角色。\n\n"
    "## 1. 全部供者数量\n\n" + markdown_table(donor_counts) +
    "\n\n## 2. 亚型汇总\n\n" + markdown_table(subtype_summary) +
    "\n\n## 3. 标签规则\n\n"
    "其他细胞沿用作者大类，T-cells按作者minor拆分为CD4、CD8、NK、NKT、Cycling T。"
    "CAF和PVL分别合并为总群，保留其他主要细胞类型。\n\n"
    "数据来源：data/raw/wu2021_scrna/Wu_etal_2021_BRCA_scRNASeq/metadata.csv；"
    "已核对计数矩阵条码列表和元数据覆盖一致，每个样本亚型唯一，逐行和全表计数相加一致，"
    "CID4535的13类数量与现有RCTD参考相同。\n"
)
(output_dir / "donor_inventory.md").write_text(report, encoding="utf-8")
provenance = {
    "metadata_file": str(metadata_path.relative_to(project_dir)),
    "metadata_sha256": hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
    "barcode_file": str(barcode_path.relative_to(project_dir)),
    "barcode_sha256": hashlib.sha256(barcode_path.read_bytes()).hexdigest(),
    "donor_id_source": "orig.ident (author sample identifier, unchanged)",
    "subtype_source": "subtype",
    "n_cells": len(metadata),
    "n_donor_ids": len(donor_counts),
    "reference_types": reference_types,
    "additional_qc": False,
    "development_sample": "CID4535",
}
(output_dir / "provenance.json").write_text(
    json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(subtype_summary)
print("保存目录：", output_dir)


,n_donor_ids,total_cells,CAFs,T cells CD8+,T cells CD4+,NK cells,NKT cells,Cycling T-cells,PVL,Cancer Epithelial,Normal Epithelial,Endothelial,Myeloid,B-cells,Plasmablasts
subtype,,,,,,,,,,,,,,,
ER+,11,38241,2573,2887,5731,640,377,186,3265,11878,2430,5206,1831,606,631
HER2+,5,19311,1449,3301,6610,453,385,188,894,1775,968,1016,1422,624,226
TNBC,10,42512,2551,5299,6890,753,360,1154,1264,10836,957,1383,6422,1976,2667


保存目录： /Users/georicl/Documents/sc_sncell_github/results/Wu2021_donor_reference_inventory
